# 01 — Data Exploration
Cornell Grasping Dataset · AI7102

This notebook sets up the environment, downloads the Cornell dataset,
and explores the data: depth images, grasp labels, occlusion maps.


In [ ]:
import os
REPO = 'https://github.com/YOUR_USERNAME/occlusion-robust-grasp-cornell.git'
if not os.path.exists('occlusion-robust-grasp-cornell'):
    os.system(f'git clone {REPO}')
os.chdir('occlusion-robust-grasp-cornell')
print('CWD:', os.getcwd())
os.system('bash scripts/setup_colab.sh')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import sys
sys.path.insert(0, '.')

from utils.cornell_loader import CornellDataset
from utils.grasp_utils import load_grasp_rectangles, build_pixel_maps
from utils.occlusion_utils import compute_occlusion_map, apply_random_occlusion, apply_occlusion_fraction
from utils.visualization import plot_sample, plot_occlusion_maps, plot_occlusion_augmentation
print('Imports OK | CUDA:', torch.cuda.is_available())

In [ ]:
# Load full dataset
ds_train = CornellDataset('data/raw/cornell', split='train')
ds_val   = CornellDataset('data/raw/cornell', split='val')
print(f'Train: {len(ds_train)} samples')
print(f'Val  : {len(ds_val)} samples')
print(f'Total: {len(ds_train) + len(ds_val)} samples')

In [ ]:
# Inspect a sample
sample = ds_train[0]
print('Input shape :', sample['input'].shape)
print('Quality shape:', sample['quality'].shape)
print('Angle shape  :', sample['angle'].shape)
print('Width shape  :', sample['width'].shape)

depth   = sample['depth_raw'].squeeze().numpy()
quality = sample['quality'].squeeze().numpy()
angle   = sample['angle'].squeeze().numpy()
width   = sample['width'].squeeze().numpy()

# Load GT rectangles
s       = ds_train.samples[0]
gt_rects = load_grasp_rectangles(s['label_path'])
print(f'GT rectangles: {len(gt_rects)}')

fig = plot_sample(depth, quality, angle, width, gt_rects=gt_rects)
os.makedirs('results', exist_ok=True)
plt.savefig('results/sample_0.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Occlusion maps
bin_map = compute_occlusion_map(depth, mode='binary')
var_map = compute_occlusion_map(depth, mode='variance', window=15)

print(f'Binary map  — missing pixels: {bin_map.mean()*100:.1f}%')
print(f'Variance map — mean: {var_map.mean():.3f}  max: {var_map.max():.3f}')

fig = plot_occlusion_maps(depth, bin_map, var_map)
plt.savefig('results/occlusion_maps.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Synthetic occlusion augmentation at different levels
fracs = [0.1, 0.2, 0.3, 0.5, 0.6]
aug_depths = [apply_occlusion_fraction(depth.copy(), f) for f in fracs]

fig = plot_occlusion_augmentation(
    depth, aug_depths,
    labels=[f'{int(f*100)}% occluded' for f in fracs]
)
plt.savefig('results/occlusion_augmentation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved results/occlusion_augmentation.png')

In [ ]:
# Dataset statistics
from utils.occlusion_utils import image_occlusion_fraction
import pandas as pd

print('Computing dataset statistics...')
stats = []
for i, s in enumerate(ds_train.samples[:100]):   # first 100
    d    = ds_train._load_depth(s['depth_path'])
    rects = load_grasp_rectangles(s['label_path'])
    stats.append({
        'n_grasps':         len(rects),
        'missing_fraction': image_occlusion_fraction(d),
    })

df = pd.DataFrame(stats)
print(df.describe().round(3))

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
df['n_grasps'].hist(ax=axes[0], bins=15, color='steelblue', edgecolor='white')
axes[0].set_title('Grasp rectangles per image'); axes[0].set_xlabel('Count')
df['missing_fraction'].hist(ax=axes[1], bins=15, color='tomato', edgecolor='white')
axes[1].set_title('Missing depth fraction'); axes[1].set_xlabel('Fraction')
plt.tight_layout()
plt.savefig('results/dataset_stats.png', dpi=150)
plt.show()
print('\n✓ Exploration done. Open 02_baseline_training.ipynb next.')